# 15.7 Reading Failures — Tracebacks, Exceptions and Logging

**Prerequisites:** 6.1 Exception Handling, 6.2 Custom Exceptions, 15.1–15.6  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The debugging loop, and where it starts: a failure you can read
- 🔴 **Fine-grained error locations** (3.11+) — the `~~~^^^` markers that name the *sub-expression*
- How chained exceptions display, and how to read a two-part traceback
- The `traceback` module — turning a failure into data you can process
- `sys.excepthook` and `threading.excepthook` — catching what nothing else catches
- 🔴 **print vs logging vs debugger** — choosing the right instrument
- `logging.exception()`, `exc_info=` and `stacklevel=`
- `faulthandler` — getting a traceback out of a **hung** or crashing process

---

## Where debugging starts

**15.1–15.6** were about finding out *that* something is wrong. This notebook and the next two
are about finding out **why**.

The two halves are one loop, which is why they share a folder:

```
   ┌─────────────────────────────────────────────────────┐
   │                                                     │
   ▼                                                     │
 a test fails  ──>  READ the failure   ──>  form a      │
 (15.1–15.6)        (15.7, here)            hypothesis   │
                          │                     │        │
                          │                     ▼        │
                          └──────>  INSPECT the state ───┘
                                    (15.8, pdb)
                                         │
                                         ▼
                            can't reproduce it? ──> 15.9, strategy
```

🔴 **The single highest-value debugging skill is reading a traceback properly**, and you
already have it from **6.1**: read bottom-up — last line is *what*, the frame above is *where*,
everything above that is *how*.

This notebook picks up where that left off. Everything below runs in a **separate interpreter**
via `subprocess`, because tracebacks printed by a notebook cell are reformatted by the notebook
and you would not see what Python actually emits.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py157_"))


def run_script(name, source, *args, flags=()):
    """Write a script, run it in a fresh interpreter, return its combined output."""
    path = WORK / name
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    done = subprocess.run([sys.executable, *flags, str(path), *args],
                          capture_output=True, text=True, encoding="utf-8",
                          errors="replace", timeout=120)
    return (done.stdout + done.stderr).rstrip()


print("scratch:", WORK)
print("python :", sys.version.split()[0])

## 🔴 Fine-grained error locations

Before Python 3.11, a traceback told you the **line**. On a line like this:

```python
return order["items"]["count"] * order["price"]["net"]
```

…`TypeError: 'NoneType' object is not subscriptable` left you guessing which of the four
subscripts was the problem.

Since **3.11**, the traceback underlines the **exact sub-expression** that raised, using two
markers:

| Marker | Means |
|---|---|
| `~~~~~` | the part of the expression that was being *evaluated* |
| `^^^^^` | the part that actually **raised** |

In [ ]:
print(run_script("locations.py", r"""
    def totals(order):
        return order["items"]["count"] * order["price"]["net"]

    print(totals({"items": {"count": 3}, "price": None}))
"""))

Read the caret line: `~~~~~~~~~~~~~~^^^^^^^` sits under
`order["price"]["net"]` — specifically under the `["net"]` subscript. It is `order["price"]`
that is `None`, and the traceback says so without you reasoning about it at all.

> **Version note.** This is 3.11+. On 3.10 and earlier you get the line only. It is one of the
> strongest practical reasons to run a modern Python: on a line with several calls or
> subscripts, it converts a five-minute question into a zero-second one.
>
> **3.13** added colour to tracebacks in the REPL and in `python file.py` when the output is a
> terminal. Piped or captured — as here — the colour is suppressed, which is why this output is
> plain.

## Chained exceptions

**6.2** covered `raise ... from`. Here is what the two forms actually *look like* when they
reach a log, because that is how you will meet them.

In [ ]:
CHAINING = r"""
    class ConfigError(Exception):
        pass


    def load_explicit(raw):
        try:
            return int(raw)
        except ValueError as exc:
            raise ConfigError("retry ceiling must be a number") from exc


    def load_implicit(raw):
        try:
            return int(raw)
        except ValueError:
            raise ConfigError("retry ceiling must be a number")


    import sys
    which = sys.argv[1]
    (load_explicit if which == "explicit" else load_implicit)("thirty")
"""

print("### raise ... from exc     (explicit: __cause__)")
print(run_script("chaining.py", CHAINING, "explicit"))
print()
print("### raise inside except    (implicit: __context__)")
print(run_script("chaining.py", CHAINING, "implicit"))

Two tracebacks, joined by a sentence — and **the sentence is the
information**:

| Joining sentence | Attribute | Means |
|---|---|---|
| *"The above exception was the direct cause of the following exception"* | `__cause__` | you wrote `from exc` — deliberate translation |
| *"During handling of the above exception, another exception occurred"* | `__context__` | it happened *while* handling — often a **bug in the handler** |

🔴 **That second one deserves suspicion.** Sometimes it is intentional. Often it means your
`except` block itself blew up — a typo in the error message, a missing variable — and the
exception you are now reading has nothing to do with the original problem. When you see
*"During handling…"*, read the **top** traceback first: that is the real failure.

To suppress the chain entirely — when the inner exception is noise the caller should never
see — use `raise ConfigError(...) from None`.

**Read both halves, always.** The most common debugging mistake after reading top-down is
reading only the *bottom* traceback of a chained pair.

## The `traceback` module

A traceback is not just text — it is an object you can inspect, format and process. That is how
error reporters, log aggregators and test frameworks work.

In [ ]:
print(run_script("tbmodule.py", r"""
    import traceback


    def parse_budget(raw):
        return int(raw)


    def load_config(settings):
        return parse_budget(settings["budget"])


    try:
        load_config({"budget": "thirty"})
    except ValueError as exc:
        print("--- just the exception, no frames ---")
        print("".join(traceback.format_exception_only(exc)).rstrip())

        print("\n--- the frames, as structured data ---")
        report = traceback.TracebackException.from_exception(exc)
        for frame in report.stack:
            name = frame.filename.rsplit("\\", 1)[-1].rsplit("/", 1)[-1]
            print(f"  {name}:{frame.lineno:<4} {frame.name+'()':<16} {frame.line}")

        print("\n  exception type :", report.exc_type_str)
        print("  message        :", str(exc))
        print("  frame count    :", len(traceback.extract_tb(exc.__traceback__)))

        print("\n--- the deepest frame is where it actually broke ---")
        deepest = report.stack[-1]
        print(f"  {deepest.name}() line {deepest.lineno}: {deepest.line}")
"""))

| Function | Gives you |
|---|---|
| `traceback.print_exc()` | the whole thing, to `stderr` — the quick one |
| `traceback.format_exc()` | the same, as a string (used in **15.1**'s runner) |
| `format_exception_only(exc)` | just `Type: message`, no frames |
| `extract_tb(exc.__traceback__)` | a list of `FrameSummary` — filename, lineno, name, line |
| `TracebackException.from_exception(exc)` | the whole thing as an object, chain included |

`TracebackException` is the one worth knowing: it captures everything **without holding a
reference to the frames**, so you can store it, send it, or format it later without keeping
every local variable alive.

> **Version note.** `TracebackException.exc_type` was deprecated in **3.13** in favour of
> **`exc_type_str`**, used above. On 3.12 and earlier, `exc_type` returns the class itself.

## When nothing catches it: `excepthook`

An exception nobody handles goes to **`sys.excepthook`**, which prints the traceback and exits.
Replace it and you control what a crash looks like — this is how crash reporters install
themselves.

In [ ]:
print(run_script("hook.py", r"""
    import sys
    import traceback


    def crash_reporter(exc_type, exc, tb):
        # A real reporter would send this somewhere. Here: a one-line summary,
        # plus the deepest frame, which is what you actually need first.
        frames = traceback.extract_tb(tb)
        where = frames[-1] if frames else None
        print("=" * 58)
        print(f"  CRASH  {exc_type.__name__}: {exc}")
        if where:
            name = where.filename.rsplit("\\", 1)[-1].rsplit("/", 1)[-1]
            print(f"  at     {name}:{where.lineno} in {where.name}()")
            print(f"  code   {where.line}")
        print("=" * 58)


    sys.excepthook = crash_reporter


    def drain_pool(pool):
        return pool["connections"].pop()


    drain_pool({"connections": []})
"""))

🔴 **`sys.excepthook` does not fire for exceptions in threads.** Those go to
**`threading.excepthook`** instead (**12.2** showed an exception in a thread vanishing
silently). If you install one and not the other, half your crashes stay invisible. There is
also `asyncio`'s loop exception handler for the third case (**12.5**).

| Escapes from | Handled by |
|---|---|
| the main thread | `sys.excepthook` |
| any other thread | `threading.excepthook` (3.8+) |
| an asyncio task | `loop.set_exception_handler(...)` |
| an unraisable context (`__del__`, GC) | `sys.unraisablehook` (3.8+) |

## 🔴 print, logging, or the debugger?

Everyone starts with `print`. That is fine — but knowing when to reach past it is most of the
skill.

| Instrument | Use when | Cost |
|---|---|---|
| **`print`** | one quick look, code you are actively editing | you must remove it; no levels; no context |
| **`logging`** | you want it to stay; production; something intermittent | a few lines of setup |
| **the debugger** (**15.8**) | you do not yet know *what* to inspect | interactive; awkward in CI |
| **a test** (**15.1**) | you want the answer checked forever | slowest to write, only one that lasts |

The honest rule: `print` is for questions you will answer in the next sixty seconds. Anything
you might want to ask **again** should be `logging`, because it can be switched on in
production without editing code.

`logging.exception()` is the one to memorise — it logs at `ERROR` **with the traceback
attached**, and it only works inside an `except` block.

In [ ]:
print(run_script("logdemo.py", r"""
    import logging

    logging.basicConfig(
        level=logging.DEBUG,
        format="%(levelname)-8s %(name)s: %(message)s",
    )
    log = logging.getLogger("worker")


    def process(job_id, raw_budget):
        # Lazy %-formatting: the string is only built if DEBUG is enabled.
        log.debug("processing %s with raw_budget=%r", job_id, raw_budget)
        try:
            return int(raw_budget)
        except ValueError:
            # .exception() == .error(..., exc_info=True). Only valid in an except block.
            log.exception("job %s could not parse its budget", job_id)
            return 0


    process("build-1", "30")
    process("build-2", "thirty")
    log.warning("finished with %d failure(s)", 1)
"""))

Three things in that output worth copying:

1. **`log.debug("... %s ...", value)` — not an f-string.** With `%s` placeholders the message
   is only formatted **if the level is enabled**. An f-string is built every time, even when
   `DEBUG` is off. On a hot path that is real cost for no benefit.
2. **`log.exception(...)` attached the full traceback** to the `ERROR` line, automatically.
   Outside an `except` block use `log.error("...", exc_info=True)`.
3. **The levels are a dial you turn in production**, not a decision you bake in:

| Level | Means | Rule of thumb |
|---|---|---|
| `DEBUG` | diagnostic detail | off in production, on when hunting |
| `INFO` | normal progress | "the service started", "job finished" |
| `WARNING` | something is off, we coped | retried, fell back, deprecated path |
| `ERROR` | this operation failed | with `exc_info` |
| `CRITICAL` | the process cannot continue | |

> **`stacklevel=2`.** Inside a logging *helper*, the record would otherwise report **your
> helper's** filename and line, not the caller's. `log.warning(msg, stacklevel=2)` blames the
> caller instead. Added in 3.8, and near-invisible until you need it.

> **Scope note.** This is logging as a *diagnostic instrument*. Handlers, formatters,
> configuration files and structured logging belong to **17 Modern Python Features**.

## Warnings as a diagnostic channel

`warnings` is the third channel, for "this ran, but you should know". Its value in debugging is
that Python emits some **for free** — and by default many are hidden.

🔴 **`python -X dev` turns on developer mode**, which surfaces `ResourceWarning` and other
diagnostics that are silent by default.

In [ ]:
LEAKY = r"""
    import os
    import tempfile

    path = os.path.join(tempfile.mkdtemp(), "spool.txt")
    handle = open(path, "w")          # never closed - no `with`
    handle.write("queued")
    del handle                        # dropped; the file object is collected
    print("finished, apparently fine")
"""

print("### python leaky.py")
print(run_script("leaky.py", LEAKY))
print()
print("### python -X dev leaky.py")
print(run_script("leaky.py", LEAKY, flags=("-X", "dev")))

The default run says *"finished, apparently fine"*. Developer mode says
`ResourceWarning: unclosed file` — a real bug (**08** covers why `with` matters), invisible
without the flag.

Useful diagnostic flags, all zero-effort:

| Flag | Surfaces |
|---|---|
| `-X dev` | `ResourceWarning`, and extra checks in the interpreter |
| `-W error` | every warning becomes an exception (**15.1**, **15.3**) |
| `-X faulthandler` | a C-level traceback on a hard crash |
| `-X importtime` | why your program takes 4 seconds to start |
| `PYTHONASYNCIODEBUG=1` | slow callbacks and un-awaited coroutines (**12.5**) |

## 🔴 `faulthandler` — when the process is *stuck*

Every technique so far assumes your program **stopped with an exception**. Some do not: an
infinite loop, a deadlock (**12.2**), a blocking socket read (**11.2**), a segfault in a C
extension. There is no traceback because nothing raised.

`faulthandler.dump_traceback_later(timeout)` sets a timer that dumps the stack of **every
thread** if the program is still running when it expires.

In [ ]:
print(run_script("hang.py", r"""
    import faulthandler
    import threading
    import time

    # If we are still alive in 1 second, dump every thread's stack and exit.
    faulthandler.dump_traceback_later(1.0, exit=True)


    def poll_queue():
        while True:                      # the bug: no exit condition
            time.sleep(0.05)


    worker = threading.Thread(target=poll_queue, name="queue-poller", daemon=True)
    worker.start()

    time.sleep(30)                       # main thread waits forever
    print("never reached")
"""))

That output names **both** stuck threads and the exact line each is sitting
on — `poll_queue` at its `time.sleep`, and `<module>` at the `time.sleep(30)`. For a hung
process that is the whole diagnosis.

| Call | Use |
|---|---|
| `faulthandler.enable()` | dump a traceback on segfault / `SIGSEGV` |
| `faulthandler.dump_traceback_later(n, exit=True)` | watchdog for hangs |
| `faulthandler.register(signal.SIGUSR1)` | dump on demand — 🔴 **Unix only**, no `SIGUSR1` on Windows |
| `python -X faulthandler script.py` | enable without touching the code |

On a hung process you have not instrumented, the other route is attaching a debugger to the
running process — that is **15.8**.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Reading a traceback top-down.** The first frame is usually an entry point that tells you nothing. Read the last line, then work up (**6.1**).
2. 🔴 **Reading only the bottom half of a chained traceback.** *"During handling of the above exception"* often means the `except` block itself is broken — the real failure is the top one.
3. **Using an f-string in a log call.** `log.debug(f"...{x}")` formats the message even when `DEBUG` is off. Use `log.debug("...%s", x)`.
4. **`log.error(str(exc))`** — throws the traceback away. Use `log.exception(...)` inside an `except` block.
5. **Installing `sys.excepthook` and forgetting `threading.excepthook`.** Every crash in a worker thread stays invisible (**12.2**).
6. **`print` debugging that gets committed.** It has no level, no timestamp, no way to switch off, and it goes to stdout where it corrupts piped output.
7. **Assuming no traceback means no bug.** A hang produces nothing at all — that is what `faulthandler` is for.
8. **Ignoring warnings.** `ResourceWarning` and `DeprecationWarning` are hidden by default; `-X dev` and `-W error` make them visible while they are still cheap to fix.
9. **Catching an exception just to `print` it and continue.** You have now hidden the failure *and* kept the broken state.

## Best Practices

- Read the last line, the deepest frame, then the chain — in that order, every time.
- Run with `-X dev` while developing; it costs nothing and surfaces real bugs.
- Prefer `raise ... from exc` so the chain says *cause* rather than *during handling*.
- Use `logging` for anything you might want to ask twice; keep `print` for the next sixty seconds.
- Log with `%s` placeholders, not f-strings, so disabled levels cost nothing.
- Install both `sys.excepthook` and `threading.excepthook` in any long-running service.
- Reach for `TracebackException` when you need to store or transmit a failure rather than print it.
- Keep a `faulthandler.dump_traceback_later` watchdog in anything that can deadlock.

## Practice Exercises

Try these before moving on.

1. Write a one-line expression with three subscripts where the middle one fails. Confirm the `^^^` markers point at it, then re-run mentally as if you were on Python 3.10 — how much longer would it take you?
2. 🔴 Write a function whose `except` block has a typo (`exc.mesage`). Trigger it and read the resulting chained traceback. Which half is the real bug?
3. Rewrite `15.1`'s `run_tests` runner to use `TracebackException` instead of `format_exc()`, and print only the deepest frame for each failure.
4. Install a `sys.excepthook` that writes crashes to a file, then prove it does **not** fire for an exception raised in a `threading.Thread`. Fix that.
5. Take a function from **04 Functions** and add `DEBUG`, `INFO` and `ERROR` logging. Run it at each level and confirm the `DEBUG` message costs nothing when disabled.
6. 🔴 Write a script that deadlocks two threads on two locks (**12.2**), then get a diagnosis out of it with `faulthandler.dump_traceback_later`.
7. Run any notebook in **08 File Handling** with `python -X dev` and see whether it emits a `ResourceWarning`.

---

## Version notes

| Version | Change |
|---|---|
| **3.13** | Tracebacks are **coloured** in the REPL and terminal output; `TracebackException.exc_type` deprecated in favour of `exc_type_str` |
| **3.12** | `sys.last_exc` added, alongside the older `sys.last_value` — used by `pdb.pm()` (**15.8**) |
| **3.11** | 🔴 **Fine-grained error locations** — the `~~~^^^` markers shown above. The single biggest traceback improvement in years |
| **3.11** | `ExceptionGroup` and `except*` — a traceback can now hold *several* exceptions at once (see **12.5** for where they come from) |
| **3.10** | Much better `SyntaxError` messages: unclosed brackets are pointed at, not just "invalid syntax" |
| **3.8** | `threading.excepthook` and `sys.unraisablehook` added; `stacklevel=` on logging calls |

## Where next

| Notebook | Covers |
|---|---|
| **15.8 The Interactive Debugger** | `pdb`, `breakpoint()`, post-mortem, and pytest's `--pdb` |
| **15.9 Debugging in Practice** | strategy, bisection, and the bugs that will not sit still |

## Related

- **6.1 Exception Handling** — reading a traceback, where this began
- **6.2 Custom Exceptions and Chaining** — `raise ... from`, `__cause__` / `__context__`
- **12.2 Threading** — the exception in a thread that nothing reports
- **15.1** — `format_exc()` in the hand-rolled runner
- **15.3 / 15.4** — `--tb=` options, and `caplog` for asserting on log output
- **17 Modern Python Features** — `logging` configuration, handlers and formatters